<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/Fine_Tuning_LLMs_QLoRA_Lab2_Simplified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning LLMs — Hands-on Lab
## Teach a small model to answer support tickets in *your* house style (QLoRA + SFT)

**Companion to:** *Fine-Tuning LLMs, Part 1 & Part 2*

---

### What you will build

A tiny open model (**Qwen2.5-0.5B-Instruct**, Apache-2.0, no login needed) that replies to customer-support
messages in **one fixed, scannable format — every single time**.

You will see the model **before** and **after** training on the same questions. That visible change is the point of the lab.

### What it costs

| | |
|---|---|
| **GPU** | Free Google Colab **T4** |
| **Paid account / API key / HF token** | Not needed |
| **Training time** | ~5–10 minutes |
| **File you keep at the end** | a "LoRA adapter" of about 50 MB |

### How to run it

1. Open this notebook in **Google Colab**.
2. `Runtime → Change runtime type → T4 GPU` → Save.
3. Run the cells one at a time from the top (or `Runtime → Run all`).
4. After the install cell, Colab may ask you to **restart the session**. Restart, then continue from the *next* cell.

### If you don't write code

You can run every cell top-to-bottom without editing anything. Your job is to **read the printed outputs** and
the comments. At the end, **Step 8 — TRY EDITING THIS** is where the real intuition comes from.

### How each code cell is documented

Every code cell starts with a comment box that tells you:

* **WHAT** — what the cell does, in plain English
* **INPUT** — what it needs (from earlier cells or from you)
* **OUTPUT** — what you should see after running it

### The steps

| # | Step | Idea |
|---|---|---|
| 1 | Build a small, perfectly consistent dataset | *Data quality decides the outcome* |
| 2 | Load the model in 4-bit | *the "Q" in QLoRA* |
| 3 | Look at the model **before** training | *Always measure a baseline first* |
| 4 | Attach LoRA adapters | *the "LoRA" in QLoRA — train only a tiny part* |
| 5 | Train with `SFTTrainer` | *SFT = Supervised Fine-Tuning* |
| 6–7 | Compare before vs after, then save the adapter | *The adapter is a file you own* |

---
## Step 0 — Install the libraries

Five libraries do all the work:

| Library | Job |
|---|---|
| `transformers` | loads the model and its tokenizer |
| `bitsandbytes` | the 4-bit compression that makes **QLoRA** possible |
| `peft` | adds the **LoRA** adapters |
| `trl` | `SFTTrainer` — the supervised fine-tuning loop |
| `datasets` | holds the training examples |

(`accelerate` is a helper that `transformers` uses behind the scenes to run on the GPU.)

> **Note:** Colab may print a dependency warning and ask you to restart the session. That is normal — restart and
> carry on from the next cell. You do **not** need to re-run this one.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Installs (or upgrades) the libraries this lab needs.
# INPUT  : nothing (needs internet)
# OUTPUT : "Installed" message. Colab may show a "Restart session" button -
#          if so, click it and continue from the NEXT cell.
# ---------------------------------------------------------------------------

# "%pip" is the notebook version of "pip install". "-q" keeps the output short.
%pip install -q -U transformers trl peft datasets accelerate bitsandbytes

print("Installed. If Colab shows a 'Restart session' button, click it, then continue from the NEXT cell.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
Installed. If Colab shows a 'Restart session' button, click it, then continue from the NEXT cell.


---
## Step 0b — Check that we have a GPU

Training a model on a normal CPU would take hours. If the next cell stops with "No GPU found", go to
**Runtime → Change runtime type → T4 GPU** and run it again.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Confirms a GPU is available before we waste time on later steps.
# INPUT  : nothing
# OUTPUT : the GPU name and its memory, e.g.  "GPU: Tesla T4 (15.6 GB)"
#          (or a clear error message if there is no GPU)
# ---------------------------------------------------------------------------

# WHY: PyTorch is the engine under every library in this lab. It runs the maths and talks to the GPU.
import torch

# This line stops the notebook with a helpful message if no GPU is found.
assert torch.cuda.is_available(), "No GPU found. Go to Runtime > Change runtime type > T4 GPU, then run again."

gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  ({gpu_memory_gb:.1f} GB)")

GPU: Tesla T4  (15.6 GB)


---
## Step 1 — The use case, and the format we want

**Nimbus** is a fictional cloud file-storage product. Its support team wants every reply to follow the same
four-part shape, so customers can scan it in three seconds:

```
Thanks for reaching out about <the topic>.

What's happening: <one sentence explaining the cause>
What to do: <one concrete action>
If that doesn't help: <the escalation path>

— Nimbus Support
```

A general-purpose model *can* answer these questions — but it will ramble, vary its structure and forget the
sign-off. **That is exactly what fine-tuning fixes.** We are not teaching it new facts; we are teaching it a *behaviour*.

### Why we build the data in code

Real projects start from exported tickets. We generate ours from templates so that:

* the notebook needs no download,
* the target format is **100% consistent** — the most important property of a training dataset,
* you can see exactly what a training example looks like.

> **Remember:** 50–500 *excellent* examples beat 5,000 sloppy ones. We use 130.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Builds 130 training examples. Each example is a short conversation:
#          system instruction + customer message + the perfect reply.
# INPUT  : the 13 support issues written below (you can read and edit them)
# OUTPUT : `rows` - a list of 130 examples, and a printout of 5 customer messages
# ---------------------------------------------------------------------------

# WHY: "random" lets us pick and shuffle wording. seed(0) makes the result identical for everyone.
import random
random.seed(0)

# The instruction we give the model at the start of every conversation.
SYSTEM = (
    "You are Nimbus Support. Reply to every customer message using exactly this format:\n"
    "Thanks for reaching out about <topic>.\n\n"
    "What's happening: <one sentence>\n"
    "What to do: <one action>\n"
    "If that doesn't help: <escalation>\n\n"
    "— Nimbus Support"
)

# Each issue = a topic, 4 ways a customer might ask about it, and the 3 parts of the answer.
ISSUES = [
    dict(topic="a password reset email that never arrived",
         asks=["I asked for a password reset 20 minutes ago and nothing has arrived.",
               "No reset email. Can you resend it?",
               "Password reset mail is not coming to my inbox.",
               "I clicked forgot password but never got the mail."],
         cause="Reset emails can take a few minutes and are often filtered into spam or promotions.",
         action="Search your mail for 'Nimbus' including the spam folder, then request a new link from the sign-in page.",
         escalate="Reply here with the email address on the account and we will send a link manually."),
    dict(topic="an upload that keeps failing",
         asks=["My uploads fail at about 80% every time.",
               "Files won't upload, it just stops halfway.",
               "Upload keeps failing with no error message.",
               "I cannot upload anything since this morning."],
         cause="Uploads usually stall when the connection drops or the file is above your plan's single-file limit.",
         action="Retry on a stable network and confirm the file is under 5 GB.",
         escalate="Send us the file name and size and we will check the upload log for your account."),
    dict(topic="being charged twice",
         asks=["I was billed twice this month.",
               "There are two charges on my card for the same subscription.",
               "Double payment taken, please refund one.",
               "My statement shows the Nimbus fee twice."],
         cause="A duplicate charge is normally a pending authorisation that has not yet dropped off the statement.",
         action="Check whether one of the two lines is still marked pending; those clear on their own within five working days.",
         escalate="If both have settled, reply with the last four digits of the card and we will refund one immediately."),
    dict(topic="cancelling the subscription",
         asks=["How do I cancel my plan?",
               "I want to stop my subscription before the next renewal.",
               "Please cancel my account.",
               "Where is the cancel button?"],
         cause="Cancellation is self-serve and takes effect at the end of the current billing period.",
         action="Open Settings, choose Billing, then Cancel plan and confirm.",
         escalate="If the option is greyed out, tell us your workspace name and we will cancel it for you."),
    dict(topic="a shared link that does not work",
         asks=["The share link I sent my client returns an error.",
               "My colleague says the shared folder link is broken.",
               "Shared link says access denied.",
               "People outside my team cannot open the link I shared."],
         cause="Share links default to team-only access and also expire after 30 days.",
         action="Open the file, choose Share, and set the link to 'Anyone with the link' with a new expiry date.",
         escalate="Paste the link here and we will check its permissions from our side."),
    dict(topic="files missing after a sync",
         asks=["Some of my files disappeared after syncing.",
               "The desktop app synced and now a folder is gone.",
               "I lost files following the last sync.",
               "My documents folder is empty after sync."],
         cause="Missing files after a sync are almost always moved rather than deleted, and stay recoverable for 30 days.",
         action="Open the Trash view in the web app and use Restore on the affected folder.",
         escalate="If nothing is in Trash, send the folder name and the approximate time and we will restore from backup."),
    dict(topic="slow download speeds",
         asks=["Downloads are extremely slow today.",
               "It takes forever to download a 200 MB file.",
               "Why is my download speed so bad?",
               "Downloading from Nimbus is much slower than usual."],
         cause="Slow transfers are usually caused by the region setting rather than by your connection.",
         action="Go to Settings, then Storage region, and pick the region closest to you.",
         escalate="If the speed does not improve, send us a screenshot of a speed test and your region setting."),
    dict(topic="enabling two-factor authentication",
         asks=["How do I turn on 2FA?",
               "I want two-factor authentication on my account.",
               "Can I add an authenticator app?",
               "Where do I enable two step login?"],
         cause="Two-factor authentication is available on every plan, including the free one.",
         action="Open Settings, then Security, choose Add authenticator app and scan the QR code.",
         escalate="If the codes are rejected, check your phone's clock is set automatically and then contact us."),
    dict(topic="storage that is full",
         asks=["I've hit my storage limit.",
               "It says my account is full and I cannot upload.",
               "Out of space on Nimbus, what now?",
               "Storage quota exceeded message keeps appearing."],
         cause="Deleted files still count towards your quota while they sit in Trash.",
         action="Empty the Trash from the web app, then check Settings for the largest folders.",
         escalate="If you are still over the limit, reply here and we will show you the upgrade options."),
    dict(topic="adding a teammate to the workspace",
         asks=["How do I invite a colleague?",
               "I need to add someone to my team.",
               "Can I give a new employee access?",
               "Where do I add users to the workspace?"],
         cause="Only workspace owners and admins can send invitations.",
         action="Open Members in the sidebar, click Invite, and enter their work email address.",
         escalate="If you cannot see the Invite button, ask your workspace owner or tell us the workspace name."),
    dict(topic="the mobile app crashing",
         asks=["The Android app crashes on launch.",
               "Nimbus closes immediately on my phone.",
               "The mobile app keeps crashing when I open a folder.",
               "App crashes every time I try to preview a photo."],
         cause="Crashes on open are almost always a stale cache left by an older app version.",
         action="Update the app from the store, then clear its cache from your phone's app settings.",
         escalate="If it still crashes, send us your phone model and app version and we will check the crash log."),
    dict(topic="exporting all of your data",
         asks=["I want to download everything I have stored.",
               "How can I export all my files at once?",
               "Is there a way to get a full backup of my account?",
               "I need a complete export before I leave."],
         cause="Full exports are prepared in the background and delivered as a download link by email.",
         action="Open Settings, then Data, and choose Request full export.",
         escalate="If the email does not arrive within 24 hours, reply here and we will regenerate the link."),
    dict(topic="restoring an older version of a file",
         asks=["Can I go back to yesterday's version of a document?",
               "I overwrote a file, is the old one recoverable?",
               "How does version history work?",
               "I need the previous version of a spreadsheet."],
         cause="Every save creates a version, and versions are kept for 30 days on all paid plans.",
         action="Right-click the file, choose Version history, and restore the timestamp you need.",
         escalate="If the version you need is older than 30 days, send us the file path and we will check backups."),
]

def answer(issue):
    """Turn one issue into the perfect 4-part reply. The format never changes."""
    return (
        f"Thanks for reaching out about {issue['topic']}.\n\n"
        f"What's happening: {issue['cause']}\n"
        f"What to do: {issue['action']}\n"
        f"If that doesn't help: {issue['escalate']}\n\n"
        f"— Nimbus Support"
    )

# Real customers do not all write the same way, so we vary the QUESTION with greetings and
# sign-offs. The ANSWER stays exactly the same. (Vary the input, hold the output shape constant.)
PREFIXES = ["", "Hi, ", "Hello — ", "Hey team, ", "Quick one: ", "Morning, "]
SUFFIXES = ["", " Thanks!", " Please advise.", " Any ideas?", " This is urgent.", " Appreciate the help."]
VARIANTS_PER_ISSUE = 10          # 13 issues x 10 variants = 130 examples

rows = []
for issue in ISSUES:
    seen = set()                 # remembers the questions already used, to avoid duplicates
    while len(seen) < VARIANTS_PER_ISSUE:
        question = random.choice(PREFIXES) + random.choice(issue["asks"]) + random.choice(SUFFIXES)
        if question in seen:
            continue
        seen.add(question)
        # One training example = system instruction + customer question + the perfect reply.
        rows.append({"messages": [
            {"role": "system",    "content": SYSTEM},
            {"role": "user",      "content": question},
            {"role": "assistant", "content": answer(issue)},
        ]})

random.shuffle(rows)             # mix the issues so training does not see them in order

print(f"Issue types         : {len(ISSUES)}")
print(f"Total examples built: {len(rows)}")
print("\nA sample of how varied the customer messages are:")
for r in rows[:5]:
    print("  -", r["messages"][1]["content"])

Issue types         : 13
Total examples built: 130

A sample of how varied the customer messages are:
  - Hello — How do I cancel my plan? Any ideas?
  - I asked for a password reset 20 minutes ago and nothing has arrived. Appreciate the help.
  - Can I go back to yesterday's version of a document?
  - Morning, The share link I sent my client returns an error.
  - Quick one: Please cancel my account. This is urgent.


### Split the data — **before** we train, not after

We hold out 15 examples the model will **never** see during training. Without them you cannot tell real
improvement from luck.

> **Be honest about what this measures.** The same *issue types* appear in both parts, just worded differently.
> So this tests **format consistency on new wordings**, not skill on issues the model has never met.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Splits the 130 examples into 115 for training and 15 for testing.
# INPUT  : `rows` from the previous cell
# OUTPUT : `train_ds` (115 examples), `eval_ds` (15 examples) and a printout
#          of one full training example so you can see what the model learns from
# ---------------------------------------------------------------------------

# WHY: "Dataset" is Hugging Face's table format - the trainer in Step 5 expects data in this shape.
from datasets import Dataset

eval_rows  = rows[:15]           # first 15 = test questions (kept secret from the model)
train_rows = rows[15:]           # remaining 115 = what the model learns from

train_ds = Dataset.from_list(train_rows)
eval_ds  = Dataset.from_list(eval_rows)

print(f"train: {len(train_ds)} examples")
print(f"held-out eval: {len(eval_ds)} examples  (the model never sees these)")

# Show one complete training example.
print("\n" + "=" * 78)
for message in train_ds[0]["messages"]:
    print(f"\n[{message['role'].upper()}]\n{message['content']}")
print("\n" + "=" * 78)

train: 115 examples
held-out eval: 15 examples  (the model never sees these)


[SYSTEM]
You are Nimbus Support. Reply to every customer message using exactly this format:
Thanks for reaching out about <topic>.

What's happening: <one sentence>
What to do: <one action>
If that doesn't help: <escalation>

— Nimbus Support

[USER]
Hey team, How do I invite a colleague? Please advise.

[ASSISTANT]
Thanks for reaching out about adding a teammate to the workspace.

What's happening: Only workspace owners and admins can send invitations.
What to do: Open Members in the sidebar, click Invite, and enter their work email address.
If that doesn't help: If you cannot see the Invite button, ask your workspace owner or tell us the workspace name.

— Nimbus Support



### What the model actually sees

The list of `messages` above is *not* what reaches the model. The tokenizer's **chat template** turns it into
one string with special marker tokens (like `<|im_start|>`) that this model family was trained with.

Never write this string by hand — every model family uses different markers. Let the tokenizer do it.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Loads the tokenizer (turns text into numbers) and shows how one
#          example looks after the chat template is applied.
# INPUT  : `train_ds` and the model name
# OUTPUT : the formatted text with <|im_start|> / <|im_end|> markers,
#          and the number of tokens it contains (a token is roughly 3/4 of a word)
# ---------------------------------------------------------------------------

# WHY: AutoTokenizer downloads the right tokenizer for whichever model name we give it.
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # the small base model: Apache-2.0 licence, no login, ~0.5 billion parameters

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# tokenize=False means "give me the text, not the numbers" so we can read it.
formatted = tokenizer.apply_chat_template(train_ds[0]["messages"], tokenize=False)
print(formatted)
print("\n--- token count for this example:", len(tokenizer(formatted)["input_ids"]))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

<|im_start|>system
You are Nimbus Support. Reply to every customer message using exactly this format:
Thanks for reaching out about <topic>.

What's happening: <one sentence>
What to do: <one action>
If that doesn't help: <escalation>

— Nimbus Support<|im_end|>
<|im_start|>user
Hey team, How do I invite a colleague? Please advise.<|im_end|>
<|im_start|>assistant
Thanks for reaching out about adding a teammate to the workspace.

What's happening: Only workspace owners and admins can send invitations.
What to do: Open Members in the sidebar, click Invite, and enter their work email address.
If that doesn't help: If you cannot see the Invite button, ask your workspace owner or tell us the workspace name.

— Nimbus Support<|im_end|>


--- token count for this example: 153


---
## Step 2 — Load the model in 4-bit (this is the "Q" in QLoRA)

A model is a huge list of numbers ("weights"). Normally each takes 16 bits of memory. **Quantisation** stores
them in 4 bits instead — about 4× smaller — so big models fit on small GPUs. `BitsAndBytesConfig` holds these settings:

| Setting | What it does |
|---|---|
| `load_in_4bit=True` | store the frozen weights in 4 bits instead of 16 |
| `bnb_4bit_quant_type="nf4"` | **NormalFloat4** — a 4-bit format designed for how neural-network weights are distributed |
| `bnb_4bit_use_double_quant=True` | compress the compression settings too — a small free saving |
| `bnb_4bit_compute_dtype` | the maths itself still runs in 16-bit, so accuracy is preserved |

The **LoRA adapters we add in Step 4 are not compressed** — they train in higher precision. We compress the
frozen knowledge and keep the part that is learning precise.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Downloads the base model and loads it onto the GPU in 4-bit form.
# INPUT  : MODEL_ID from the previous cell
# OUTPUT : `model` ready on the GPU, and the GPU memory it uses (about 0.5 GB)
# ---------------------------------------------------------------------------

# WHY: AutoModelForCausalLM = a model that writes text one token at a time.
#      BitsAndBytesConfig = the 4-bit compression settings (needs the bitsandbytes library).
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

DTYPE = torch.float16            # 16-bit numbers: safe on the free Colab T4 GPU

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                   # compress weights to 4 bits
    bnb_4bit_quant_type="nf4",           # the 4-bit format designed for neural networks
    bnb_4bit_use_double_quant=True,      # compress the compression constants as well
    bnb_4bit_compute_dtype=DTYPE,        # do the calculations in 16-bit
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,      # apply the 4-bit settings above
    device_map="auto",                   # put the model on the GPU automatically
    dtype=DTYPE,
)

print(f"Model loaded. GPU memory in use: {torch.cuda.memory_allocated()/1e9:.2f} GB")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. GPU memory in use: 0.46 GB


---
## Step 3 — The baseline: what does it say **before** training?

**Do not skip this.** Recording the "before" is the difference between an experiment and a guess.

Watch for: the model knows plenty about password resets and uploads — it is *not* ignorant. It simply
answers in a shape nobody asked for: the length varies, labels are missing, the sign-off is absent.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Defines `generate()` (ask the model a question, get a reply) and uses
#          it on 3 test questions BEFORE any training.
# INPUT  : `model`, `tokenizer`, and 3 questions taken from the held-out test set
# OUTPUT : for each question, the model's untrained reply. These are saved in
#          `before` so we can compare them after training.
# ---------------------------------------------------------------------------

def generate(model, user_message, max_new_tokens=180):
    """Send one customer message to the model and return its reply as text."""
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_message}]

    # Apply the chat template. add_generation_prompt=True adds the "assistant, your turn" marker.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)   # text -> numbers, on the GPU

    with torch.no_grad():                           # we are not training here, so save memory
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,          # stop after this many tokens
            do_sample=False,                        # always pick the most likely next token -> same answer every run
            pad_token_id=tokenizer.pad_token_id,    # padding marker (silences a harmless warning)
        )

    # `output` contains the prompt AND the reply. Cut the prompt off and turn the numbers back into text.
    reply_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(reply_tokens, skip_special_tokens=True).strip()


# Three held-out questions that we will ask again after training.
demo_questions = [eval_rows[i]["messages"][1]["content"] for i in range(3)]

before = {}                      # saves each question -> untrained reply
model.eval()                     # "eval mode" = we are using the model, not training it
for q in demo_questions:
    before[q] = generate(model, q)
    print("=" * 78)
    print(f"QUESTION: {q}\n")
    print(before[q])
    print()

QUESTION: Hello — How do I cancel my plan? Any ideas?

Thanks for reaching out about cancelling your plan. We can assist you with that.

What's happening: You have a plan and need to cancel it.
What to do: Contact our support team to request cancellation.
If that doesn't work, escalate the issue to our escalation process.

QUESTION: I asked for a password reset 20 minutes ago and nothing has arrived. Appreciate the help.

Thanks for reaching out about resetting your password. We're currently processing it, but we'll get back to you shortly.

What's happening: The password reset process is ongoing.
What to do: Check your email for updates or contact support if needed.
If that doesn't help: Escalate.

QUESTION: Can I go back to yesterday's version of a document?

Thanks for your question regarding the ability to return to a previously saved version of a document. Unfortunately, we cannot assist with that task as it goes beyond our current support capabilities.

However, if you have any o

### Score the baseline with a rule, not a feeling

Our target format is mechanical, so we can check it mechanically. Five markers must appear in the reply.
This is a tiny **evaluation harness** — a fair, repeatable way to measure "better".

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Builds a scoring rule (what fraction of the 5 required phrases appear?)
#          and scores the untrained model on all 15 held-out questions.
# INPUT  : `model`, `eval_rows` (15 test questions)
# OUTPUT : one line, e.g.  "BEFORE fine-tuning   format score: 0.48   (0/15 perfect)"
#          Score 1.00 = every reply has all 5 phrases. 0.00 = none of them.
# ---------------------------------------------------------------------------

# The five phrases every correct reply must contain.
MARKERS = ["Thanks for reaching out about", "What's happening:", "What to do:", "If that doesn't help:", "— Nimbus Support"]

def format_score(text):
    """Return 0.0 to 1.0: the share of the 5 required phrases found in the reply."""
    return sum(marker in text for marker in MARKERS) / len(MARKERS)

def evaluate(model, rows, label):
    """Ask the model every question in `rows`, score each reply, print and return the average."""
    scores = []
    for row in rows:
        question = row["messages"][1]["content"]
        scores.append(format_score(generate(model, question)))
    average = sum(scores) / len(scores)
    perfect = sum(score == 1.0 for score in scores)
    print(f"{label:<20} format score: {average:.2f}   ({perfect}/{len(scores)} perfect)")
    return average

baseline_score = evaluate(model, eval_rows, "BEFORE fine-tuning")

BEFORE fine-tuning   format score: 0.48   (1/15 perfect)


---
## Step 4 — Attach the LoRA adapters (the "LoRA" in QLoRA)

Instead of changing the whole model, LoRA leaves the original weights `W` **frozen** and adds two thin matrices
`A` and `B` beside them:

$$W' = W + \frac{\alpha}{r} \cdot B A$$

Only `A` and `B` are trained. During training the model adjusts their numbers to reduce its mistakes (the "loss").

| Argument | Value | Why |
|---|---|---|
| `r` | `16` | "rank" = adapter size / capacity. 8–16 is plenty for a style/format task |
| `lora_alpha` | `32` | scaling factor; using `2 × r` is a common rule of thumb |
| `lora_dropout` | `0.05` | light protection against memorising a small dataset |
| `target_modules` | attention + MLP layers | *which* layers receive adapters (details below) |
| `task_type` | `CAUSAL_LM` | we are training a text generator |

**About `target_modules`:** the names are the model's internal layers. `q_proj, k_proj, v_proj, o_proj` are the
**attention** layers (how words relate to each other). `gate_proj, up_proj, down_proj` are the **MLP** layers
(further processing of each word). Adapting both works well and is the common choice from the QLoRA paper.

**Watch the printed percentage.** Under 2% of the model is trainable — that is why this runs on a free GPU.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Prepares the 4-bit model for training and attaches the LoRA adapters.
# INPUT  : the 4-bit `model` from Step 2
# OUTPUT : `model` now has adapters. Prints e.g.
#          "trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497"
# ---------------------------------------------------------------------------

# WHY: LoraConfig = the adapter settings. get_peft_model = attaches them.
#      prepare_model_for_kbit_training = standard clean-up needed before training a 4-bit model.
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Standard QLoRA preparation. Gradient checkpointing trades a little speed for much lower GPU memory.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_config = LoraConfig(
    r=16,                        # adapter size
    lora_alpha=32,               # adapter strength
    lora_dropout=0.05,           # light regularisation
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",       # attention layers
                    "gate_proj", "up_proj", "down_proj"],         # MLP layers
)

model = get_peft_model(model, lora_config)    # attach the adapters
model.print_trainable_parameters()            # how much of the model will actually be trained

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


---
## Step 5 — Train with `SFTTrainer`

`SFTTrainer` handles the chat template, tokenisation, batching and the training loop. You supply the data
and a handful of settings:

| Argument | Value | Why |
|---|---|---|
| `learning_rate` | `2e-4` | how big each learning step is. LoRA likes larger steps than full fine-tuning |
| `num_train_epochs` | `3` | how many times the model reads the whole dataset. More than this risks memorising |
| `per_device_train_batch_size` × `gradient_accumulation_steps` | `2 × 4 = 8` | the GPU holds 2 examples at a time; it collects 4 rounds before each update, like a batch of 8 |
| `max_length` | `768` | longest example allowed (in tokens); longer costs much more memory |

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Trains the LoRA adapters on our 115 examples (the actual fine-tuning).
# INPUT  : `model` (with adapters), `train_ds`, `tokenizer`
# OUTPUT : a progress bar and a log table. The "loss" column should FALL as
#          training proceeds - that means the model is learning the format.
#          Takes roughly 5-10 minutes on a free T4.
# ---------------------------------------------------------------------------

# WHY: SFTTrainer = the training loop. SFTConfig = its settings.
from trl import SFTTrainer, SFTConfig

# WHY: "os" lets us set one environment variable, below.
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"   # use 16-bit maths on the T4 (this setting is what worked in testing)

sft_config = SFTConfig(
    output_dir="nimbus-support-lora",       # folder for the results
    num_train_epochs=3,                     # read the dataset 3 times
    per_device_train_batch_size=2,          # examples held on the GPU at once
    gradient_accumulation_steps=4,          # collect 4 rounds, then update -> effective batch of 8
    learning_rate=2e-4,                     # size of each learning step
    lr_scheduler_type="cosine",             # slowly shrink the learning step over time
    warmup_steps=5,                         # start gently for the first 5 steps
    max_length=768,                         # longest example allowed, in tokens
    optim="paged_adamw_8bit",               # memory-friendly optimiser (the "learning rule")
    gradient_checkpointing=True,            # save GPU memory
    gradient_checkpointing_kwargs={"use_reentrant": False},   # required when using PEFT/LoRA
    fp16=False, bf16=False,                 # precision is handled by the environment variable above
    logging_steps=5,                        # print the loss every 5 steps
    save_strategy="no",                     # no checkpoints - we save the adapter ourselves in Step 7
    report_to="none",                       # no online tracking tools
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

print("Starting training - roughly 5 to 10 minutes on a free T4.\n")
trainer.train()
print("\nDone.")

Tokenizing train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

Starting training - roughly 5 to 10 minutes on a free T4.



NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

### Reading the training log

* **`loss` should fall** over the run. A flat loss usually means the learning rate is too low.
* A loss that drops to almost zero on a small dataset means the model is **memorising**, not learning.

---
## Step 6 — The same questions, after training

This is the moment the lab is built around. Same model, same questions, same settings —
only the adapter is new.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Asks the trained model the same 3 questions as before and shows the
#          old and new replies side by side.
# INPUT  : the trained `model`, `demo_questions`, and `before` (saved in Step 3)
# OUTPUT : for each question, the BEFORE reply and the AFTER reply. The AFTER
#          reply should follow the 4-part Nimbus format with the sign-off.
# ---------------------------------------------------------------------------

model.eval()                          # back to "using" mode
model.config.use_cache = True         # turn speed-up memory back on for faster writing

for q in demo_questions:
    after = generate(model, q)
    print("=" * 78)
    print(f"QUESTION: {q}\n")
    print("--- BEFORE ---------------------------------------------------------")
    print(before[q])
    print("\n--- AFTER ----------------------------------------------------------")
    print(after)
    print()

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Scores the trained model on all 15 held-out questions and compares
#          it with the baseline score from Step 3.
# INPUT  : the trained `model`, `eval_rows`, `baseline_score`
# OUTPUT : Baseline, Tuned and Change scores. A big jump toward 1.00 is the
#          proof that fine-tuning worked.
# ---------------------------------------------------------------------------

tuned_score = evaluate(model, eval_rows, "AFTER fine-tuning")

print()
print(f"Baseline : {baseline_score:.2f}")
print(f"Tuned    : {tuned_score:.2f}")
print(f"Change   : {tuned_score - baseline_score:+.2f}")
print("\n(Format compliance on 15 questions the model never saw during training.)")

---
## Step 7 — Save the adapter

The adapter is a **file you own**. Compare its size with the base model it modifies — this is the
"one base model, many adapters" idea made concrete.

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Saves the trained adapter (not the whole model) to a folder and lists its files.
# INPUT  : the trained `model` and `tokenizer`
# OUTPUT : the files in the folder with their sizes. The adapter is only tens of MB,
#          while the base model is about 1 GB.
# ---------------------------------------------------------------------------

# WHY: "os" lets us list the saved files and read their sizes.
import os

SAVE_DIR = "nimbus-support-lora/adapter"
model.save_pretrained(SAVE_DIR)        # saves ONLY the small adapter weights
tokenizer.save_pretrained(SAVE_DIR)    # saves the tokenizer next to it, so the folder is self-contained

total_bytes = 0
print(f"Contents of {SAVE_DIR}:")
for filename in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, filename))
    total_bytes += size
    print(f"  {filename:<32} {size/1e6:8.2f} MB")

print(f"\nAdapter total: {total_bytes/1e6:.1f} MB")
print("Base model   : about 1 GB in 16-bit  (Qwen2.5-0.5B)")
print("A 7B base model would be about 14 GB - and its adapter would still be only tens of MB.")

### How you would load it again later

You do **not** ship the whole model — you ship the base model's name plus this small folder.

```python
from transformers import AutoModelForCausalLM
from peft import PeftModel

base  = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
tuned = PeftModel.from_pretrained(base, "nimbus-support-lora/adapter")
```

To download the adapter out of Colab:

```python
from google.colab import files
!zip -r adapter.zip nimbus-support-lora/adapter
files.download("adapter.zip")
```

---
## Step 8 — **TRY EDITING THIS** (the part that teaches the most)

Pick one. Each takes a few minutes and each demonstrates a real lesson.

| # | Change | What you should observe | Concept |
|---|---|---|---|
| 1 | Change the format in `answer()` (Step 1) — e.g. add a `Reference:` line or drop the sign-off, then re-run everything from the dataset cell | The model learns the *new* shape just as reliably. **You control the behaviour through the data.** | Data is the curriculum |
| 2 | Set `num_train_epochs=10` | Training loss drops near zero, but replies get rigid and may reuse the wrong issue's text | Overfitting |
| 3 | Set `r=4` and `lora_alpha=8` | Often still works. Size is rarely the bottleneck for a format task | Rank is not a magic knob |
| 4 | Cut the dataset to 20 examples: `train_ds = Dataset.from_list(train_rows[:20])` | The score drops sharply | Data quantity |
| 5 | Ask an **off-topic** question (the cell below) | It may now answer in support format anyway | Catastrophic forgetting |
| 6 | Swap `MODEL_ID` to `"Qwen/Qwen3-0.6B"` or `"HuggingFaceTB/SmolLM2-360M-Instruct"` | Same code, different base model | Portability of the recipe |

In [ ]:
# ---------------------------------------------------------------------------
# WHAT   : Exercise 5 - asks the trained model a question that has nothing to do
#          with support tickets.
# INPUT  : the trained `model`
# OUTPUT : the model's answer. Does it still answer normally, or does it force
#          the Nimbus support format onto a geography question?
# ---------------------------------------------------------------------------

print(generate(model, "What is the capital of France?"))

---
## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `CUDA out of memory` | batch or sequence length too large | `Runtime → Restart session`, then set `per_device_train_batch_size=1` and `gradient_accumulation_steps=8` |
| "No GPU found" / training crawls | CPU runtime | `Runtime → Change runtime type → T4 GPU` |
| `ImportError` after installing | Colab needs a restart | Restart the session, then re-run from the *GPU check* cell (skip the install) |
| `TypeError: ... unexpected keyword argument` | a library changed a setting name | Check the docs for the version you installed |
| Loss stays flat | learning rate too low, or adapters not attached | Confirm Step 4 printed a non-zero trainable count; try `learning_rate=3e-4` |
| Replies repeat one issue's text | over-training or too little variety | Fewer epochs, or add more distinct issues to `ISSUES` |

---

### Reference links

* Hugging Face PEFT — <https://huggingface.co/docs/peft>
* TRL (SFT, DPO, GRPO trainers) — <https://huggingface.co/docs/trl>
* LoRA paper — arXiv 2106.09685 · QLoRA paper — arXiv 2305.14314